<div style="padding: 1em 0.5em; color: #fff; background-color: #0969da; font-size: 1.2em;">
    Jour 1a — Introduction à la biologie moleculaire et préparation des données
</div>
<div style="border-left: 2px solid #0969da; min-height: 1.5em;margin-left: 1em;padding: 1em;">
    - Comprendre la différence entre ADN codant (CDS) et non-codant, et formuler la tâche : prédire si une fenêtre de 200 pb est codante<br>
    - Explorer les fichiers génomiques bruts (FASTA + annotations GFF) et voir comment on en extrait des fenêtres étiquetées<br>
    - Construire les splits train/val/test avec build_dataset.py et vérifier l'équilibre des classes<br>
</div>

<div style="border-left: 4px solid #0969da; background-color: #f2f6fd; padding: 0.9em 1.1em; color: #24292f;">
  <strong style="color: #0969da;">Version corrigée &mdash; solution de référence</strong><br>
  Ce notebook contient <strong>toutes les réponses</strong>. Il n'est pas à compléter et n'est pas à rendre.<br>
  Travaillez d'abord sur la version étudiante du dossier parent, et ne consultez ce fichier qu'après avoir essayé par vous-même.
</div>

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

#### **ADN codant vs. non-codant**

Un génome bactérien est une seule et longue chaîne d'ADN. Une partie seulement de cette
chaîne **code réellement pour une protéine** (une *CDS*, séquence codante — lue par
triplets appelés codons, chacun spécifiant un acide aminé). Le reste est **non-codant** :
régions intergéniques, séquences régulatrices, etc.

Notre tâche cette semaine : **étant donné une courte fenêtre d'ADN, prédire si elle se
trouve à l'intérieur d'une CDS ou non.**

C'est un problème réel et utile — la recherche de gènes (*gene finding*) est une tâche
fondamentale en bioinformatique, et elle est binaire, ce qui la rend abordable pour un
premier projet.

<img src="https://raw.githubusercontent.com/Genereux-akotenou/EEIA-bioAI-Workshop-project/main/day1/assets/illustration0.png"/>

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

#### **Regardons d'abord les fichiers bruts**

Pas de prétraitement pour l'instant. Chaque génome dans `data/supervised/raw/` est une
paire de fichiers :
- un fichier **FASTA** : la séquence brute du génome
- un fichier **GFF** : les annotations — où se trouvent les gènes/régions CDS, sur quel brin

Regardons les deux directement.

In [6]:
import sys
sys.path.append("src")

RAW_DIR = "../2-data/raw"

In [3]:
# on regarde un fichier FASTA brut du génome
fasta_path = f"{RAW_DIR}/train/GCA_000240015.1_ASM24001v1.fasta"
with open(fasta_path) as f:
    for _ in range(6):
        print(f.readline().rstrip())

>CP003195.1 Propionibacterium acnes TypeIA2 P.acn33, complete genome
CTAGCGTTCAGGGGAGTGGTACATCGTGGGTAGCTTGTCACACCGACTGTGGAAAACTGTGTGGACAACTCTCGACAACT
TCTAGAGAACAGGTGGTGGAATGTCCGACACACCGTTCGGCGACGCCGACCACCCTCGTCCCGCGCCGATCCATCCGGAT
GCTGTATTGCCCCCGCCGATGAGTTCACAGTCGGCTGATAATGATCCCACTGAGGCCCTCAATGAAGCTTGGACCAACAT
CCTTACGAAGGTTTCGAAACCTAATCGCGCGTGGTTATCCAACACTACCCCGGTGACGATGCACTCTTCTACGGCGATGG
TTGCCGTTCCCAACGAATTTGCCCGTGACCGTCTTGAATCAAAGATGCGTTACGAACTAGAAGAACTCCTCTCTGACCAT


In [4]:
# on regarde son fichier d'annotation GFF — notez les lignes CDS (colonnes : seqid, source,
# type, start, end, score, strand, phase, attributes)
gff_path = f"{RAW_DIR}/train/GCA_000240015.1_ASM24001v1.gff"
with open(gff_path) as f:
    lines = [l for l in f if not l.startswith("#")]
for l in lines[:8]:
    print(l.rstrip())

CP003195.1	Genbank	region	1	2489623	.	+	.	ID=CP003195.1:1..2489623;Dbxref=taxon:1114966;Is_circular=true;Name=ANONYMOUS;gbkey=Src;genome=chromosome;mol_type=genomic DNA;strain=P.acn33;sub-species=acnes
CP003195.1	Genbank	gene	101	1603	.	+	.	ID=gene-TIA2EST2_00005;Name=TIA2EST2_00005;gbkey=Gene;gene_biotype=protein_coding;locus_tag=TIA2EST2_00005
CP003195.1	Genbank	CDS	101	1603	.	+	0	ID=cds-AEW78118.1;Parent=gene-TIA2EST2_00005;Dbxref=NCBI_GP:AEW78118.1;Name=AEW78118.1;Note=COG0593 ATPase involved in DNA replication initiation;gbkey=CDS;locus_tag=TIA2EST2_00005;product=chromosomal replication initiator protein DnaA;protein_id=AEW78118.1;transl_table=11
CP003195.1	Genbank	gene	1991	3235	.	+	.	ID=gene-TIA2EST2_00010;Name=TIA2EST2_00010;gbkey=Gene;gene_biotype=protein_coding;locus_tag=TIA2EST2_00010
CP003195.1	Genbank	CDS	1991	3235	.	+	0	ID=cds-AEW78119.1;Parent=gene-TIA2EST2_00010;Dbxref=NCBI_GP:AEW78119.1;Name=AEW78119.1;Note=COG0592 DNA polymerase sliding clamp subunit (PCNA homolog);gb

#### **Des fichiers bruts à un jeu de données étiqueté**

`src/build_dataset.py` est le script fourni qui transforme cela en quelque chose sur lequel
on peut entraîner un modèle. Pour chaque génome, il :

1. Lit tous les intervalles CDS du GFF (`type == "CDS"`, colonnes `start`/`end`/`strand`).
2. Fait glisser des fenêtres de longueur fixe (200 pb) **à l'intérieur** des régions CDS →
   étiquette `1` (codant). Si la CDS est sur le brin `-`, la fenêtre est d'abord retournée
   en complément inverse, pour que le modèle voie toujours la séquence dans l'orientation
   codante.
3. Calcule les **intervalles** entre les régions CDS (tout ce qui n'est couvert par aucune
   CDS) et fait glisser les mêmes fenêtres dessus → étiquette `0` (non-codant).
4. Équilibre les classes par génome, mélange, et écrit un CSV par split.

<img src="https://raw.githubusercontent.com/Genereux-akotenou/EEIA-bioAI-Workshop-project/main/day1/assets/illustration1.png"/>

Il a déjà été exécuté une fois (`data/processed/{train,val,test}.csv`
existent). Si vous voulez changer la taille de fenêtre ou le ré-échantillonnage,
relancez-le :

> ```bash
> python ../../code/src/build_dataset.py --raw_dir ../../data/raw --out_dir ../../data/processed --window 200 --stride 200
```

In [5]:
from data import load_all

splits = load_all("../../2-data/processed")
for name, df in splits.items():
    print(name, df.shape, "class balance:", df["label"].value_counts().to_dict())

splits["train"].head()

train (58552, 9) class balance: {0: 29276, 1: 29276}
val (10374, 9) class balance: {0: 5187, 1: 5187}
test (18448, 9) class balance: {0: 9224, 1: 9224}


,id,split,organism,seqid,start,end,strand,label,sequence
0,train_0,train,GCA_002863665.1_ASM286366v1,CP024056.1,2697518,2697718,+,0,CAGAAAATATACCGCCAGGCATAAGTATCATGTGAAATCCAACTAT...
1,train_1,train,GCA_027912575.1_ASM2791257v1,CP115712.1,1935291,1935491,-,1,CTCAACAGGGTCTGCCGAAAGGCGGCGCTGTCGGTTTTGAACTGCT...
2,train_2,train,GCA_043159985.1_ASM4315998v1,CP150937.1,322096,322296,+,0,AGCGACAGTATTTTCTGCCGGGAGTATAGGCAGTATGGCGTTTTTT...
3,train_3,train,GCA_002442855.1_ASM244285v1,CP017306.1,76513,76713,-,1,TGATCGAGGACCAAGTGATCCTCGGCGCCGGCAGCCTGGTGCCGCC...
4,train_4,train,GCA_027912575.1_ASM2791257v1,CP115712.1,4421120,4421320,+,1,ATTGAGAACGTGTTTGAACTGGAATCGCGCACCGTGCCGTCGTCGA...


#### **Point de contrôle**

Vous devriez avoir des DataFrames `train`, `val`, `test`, chacun avec une colonne
`sequence` (la fenêtre brute de 200 pb) et une colonne `label` (1 = codant, 0 = non-codant),
à peu près équilibrées entre les classes.

Suite : `01_kmer_and_cnn_baselines.ipynb` — construire les premiers modèles à partir de ces
données.

<div style="margin-top: 3em;">
  <div style="height: 3px; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>
  <div style="display: flex; align-items: center; gap: 0.9em; padding: 1.1em 1em; font-size: 0.9em; color: #57606a; background-color: #f2f6fd; border-radius: 0 0 6px 6px;">
    <svg width="34" height="34" viewBox="0 0 34 34" fill="none" style="flex: 0 0 auto;">
      <path d="M9 3c0 7 16 7 16 14S9 24 9 31" stroke="#0969da" stroke-width="2" stroke-linecap="round"/>
      <path d="M25 3c0 7-16 7-16 14s16 7 16 14" stroke="#0969da" stroke-width="2" stroke-linecap="round" opacity="0.45"/>
      <circle cx="17" cy="10" r="1.8" fill="#0969da"/>
      <circle cx="17" cy="24" r="1.8" fill="#0969da"/>
    </svg>
    <div style="flex: 1 1 auto;">
      <div style="color: #0969da; font-weight: 600; letter-spacing: 0.03em;">Fin du Jour 1a</div>
      <div>Prochaine &eacute;tape &rarr; <code>01_kmer_and_cnn_baselines.ipynb</code></div>
    </div>
    <div style="flex: 0 0 auto; text-align: right; border-right: 2px solid #0969da; padding-right: 0.9em;">
      <div style="font-weight: 600; color: #24292f;">EEIA &middot; bioAI Workshop</div>
      <div style="font-size: 0.85em;">Semaine 4 &mdash; De l'ADN aux mod&egrave;les compress&eacute;s</div>
    </div>
  </div>
</div>